# Nettoyage et normalisation d'un fichier d'état civil    (2 heures)

## Objectif

L'objectif de ce notebook est de nettoyer automatiquement un fichier d'état civil avant son exploitation.

Les traitements portent principalement sur :

- la suppression des lignes inutiles ;
- la correction des noms et prénoms ;
- la normalisation des dates ;
- la normalisation des lieux de naissance ;
- la correction automatique des villes à l'aide d'un référentiel ;
- la génération d'un fichier final propre.

# **Correction des noms et prénoms**

## 1. Chargement des données

Cette étape consiste à importer le fichier Excel contenant les données à traiter ainsi que les bibliothèques Python nécessaires au nettoyage.

## 2. Correction des noms et prénoms

L'une des difficultés rencontrées dans le fichier est que certains enregistrements ne respectent pas la séparation entre le **nom** et le **prénom**.

Par exemple, au lieu d'avoir :

| Nom | Prénom |
|------|---------|
| DUPONT | Jean |

on peut rencontrer :

| Nom |
|------|
| DUPONT Jean |

ou encore :

| Nom |
|------|
| DUPONT MARTIN Pierre |

Dans ces cas, le nom et le prénom sont regroupés dans une seule colonne.

### Principe utilisé

Le script s'appuie sur une convention très fréquente dans les documents administratifs :

- le **nom de famille** est généralement écrit en **MAJUSCULES** ;
- le **prénom** est écrit avec une majuscule suivie de minuscules.

Le traitement consiste donc à parcourir le texte mot par mot.

Tant que les mots sont entièrement écrits en majuscules, ils sont considérés comme faisant partie du **nom**.

Dès qu'un mot n'est plus entièrement en majuscules, il est considéré comme le début du **prénom**.

### Exemple 1

Entrée :

```text
DUPONT Jean
```

Résultat :

```text
Nom     : DUPONT
Prénom  : Jean
```



### Exemple 2

Entrée :

```text
BEN ALI Mohamed
```

Résultat :

```text
Nom     : BEN ALI
Prénom  : Mohamed
```

### Pourquoi cette méthode ?

Cette règle permet de corriger automatiquement une grande partie des enregistrements où le nom et le prénom ont été saisis dans la même colonne.

Le script replace ensuite les informations dans les colonnes **Nom** et **Prénom**, ce qui facilite les traitements suivants et améliore la qualité globale des données.

In [80]:
import pandas as pd
import re

SRC = "/content/test_lieux_naissance (1).xlsx"

# Lire directement la feuille "Données brutes"
df = pd.read_excel(SRC, sheet_name="Données brutes")

# ---------------------------------------------------------------
# 1) NOM / PRENOM
# ---------------------------------------------------------------
def split_nom_prenom(nom_raw, prenom_raw):
    nom_raw = str(nom_raw).strip() if pd.notna(nom_raw) else ""
    prenom_raw = str(prenom_raw).strip() if pd.notna(prenom_raw) else ""

    # Le prénom est déjà renseigné
    if prenom_raw:
        return (
            nom_raw.upper(),
            prenom_raw,
            "NOM/PRÉNOM déjà séparés."
        )

    # Nom en MAJUSCULES (plusieurs mots possibles), puis prénom
    pattern = r'^([A-ZÀ-ÖØ-Þ\-]+(?:\s+[A-ZÀ-ÖØ-Þ\-]+)*)\s+(.+)$'
    match = re.match(pattern, nom_raw)

    if match:
        nom = match.group(1).strip()
        prenom = match.group(2).strip()

        return (
            nom,
            prenom,
            "PRÉNOM manquant dans la colonne C : déduit du contenu de la colonne B."
        )

    return (
        nom_raw.upper(),
        "",
        "PRÉNOM introuvable."
    )

## 4. Normalisation des dates

Les dates présentes dans le fichier n'étaient pas toutes écrites de la même manière. Avant de les utiliser, il était donc nécessaire de les convertir vers un format unique : **JJ/MM/AAAA**.

Le script traite automatiquement les principaux cas rencontrés.

### Cas n°1 : Mois écrit en toutes lettres

Certaines dates utilisent le nom du mois au lieu de son numéro.

Par exemple :

```text
14 jan 1839
```

Le script utilise un dictionnaire contenant les noms et les abréviations des mois pour convertir automatiquement le mois en numéro.

Résultat :

```text
14/01/1839
```

Cette méthode fonctionne également avec plusieurs variantes, par exemple :

| Date d'origine | Date obtenue |
|----------------|--------------|
| 14 janvier 1839 | 14/01/1839 |
| 14 janv. 1839 | 14/01/1839 |
| 14 fév 1839 | 14/02/1839 |
| 14 septembre 1839 | 14/09/1839 |

---

### Cas n°2 : Séparateurs différents

Les dates peuvent être écrites avec différents séparateurs :

```text
14-01-1839
```

ou

```text
14.01.1839
```

Le script remplace automatiquement les caractères `-` et `.` par `/`.

Par exemple :

```text
14-01-1839
```

devient :

```text
14/01/1839
```

---

### Cas n°3 : Année placée en premier

Certaines dates sont enregistrées sous la forme :

```text
1839/01/14
```

Le script reconnaît que les quatre premiers chiffres correspondent à l'année et réorganise automatiquement la date.

Résultat :

```text
14/01/1839
```

---

### Cas n°4 : Jour et mois inversés

Il arrive que le jour et le mois soient inversés.

Par exemple :

```text
05/14/1839
```

Le nombre **14** ne peut pas représenter un mois.

Le script détecte ce cas et échange automatiquement les deux valeurs.

Résultat :

```text
14/05/1839
```

---

### Cas n°5 : Format inconnu

Si la date ne correspond à aucun des formats prévus, aucune modification n'est effectuée.

Le script conserve alors la valeur d'origine et ajoute un commentaire indiquant qu'une vérification manuelle est nécessaire.

---

### Objectif

Grâce à ces différentes règles, les dates sont homogénéisées dans un format unique (**JJ/MM/AAAA**), ce qui facilite les recherches, les tris et les traitements réalisés par la suite.

In [81]:
MONTHS = {
    'jan': '01', 'janv': '01', 'janvier': '01',
    'fev': '02', 'fév': '02', 'fevr': '02', 'février': '02', 'fevrier':'02',
    'mar': '03', 'mars': '03',
    'avr': '04', 'avril': '04',
    'mai': '05',
    'juin': '06',
    'juil': '07', 'juillet': '07',
    'aou': '08', 'août': '08', 'aout': '08',
    'sep': '09', 'sept': '09', 'septembre': '09',
    'oct': '10', 'octobre': '10',
    'nov': '11', 'novembre': '11',
    'dec': '12', 'déc': '12', 'décembre': '12', 'decembre': '12',
}

def parse_date(raw):
    raw = str(raw).strip()
    comment = ""
    # cas avec un mois en toutes lettres : "14 jan 1839"
    m = re.match(r'^(\d{1,2})\s+([a-zA-Zéûô]+)\.?\s+(\d{4})$', raw)
    if m:
        day, mon, year = m.groups()
        mon_key = mon.lower().rstrip('.')
        if mon_key in MONTHS:
            return f"{int(day):02d}/{MONTHS[mon_key]}/{year}", "Mois en toutes lettres converti (dictionnaire mois)."
        return raw, "Mois non reconnu, à vérifier manuellement."

    # normaliser séparateurs . et - en /
    norm = raw.replace('.', '/').replace('-', '/')
    parts = norm.split('/')
    if len(parts) != 3:
        return raw, "Format de date non reconnu, à vérifier manuellement."
    a, b, c = parts
    # cas année en premier : YYYY/MM/DD
    if len(a) == 4:
        year, month, day = a, b, c
        comment = "Format année-mois-jour réordonné en JJ/MM/AAAA."
    else:
        day, month, year = a, b, c
        comment = "Séparateurs normalisés en JJ/MM/AAAA." if raw != f"{day}/{month}/{year}" else "Date déjà au bon format."
    day, month = int(day), int(month)
    # jour/mois inversés si mois > 12
    if month > 12 and day <= 12:
        day, month = month, day
        comment += " Jour et mois étaient inversés (mois>12) : permutés."
    return f"{day:02d}/{month:02d}/{year}", comment

# **Base de données extraite du Web contenant l'ensemble des villes de la Tunisie et leurs codes postaux**

In [82]:
!pip install pdfplumber openpyxl

In [83]:
import pdfplumber
import pandas as pd
import re

pdf_tunisie = pdfplumber.open("/content/ReseauCommercialSICAVTANIT.pdf")

rows = []
for page in pdf_tunisie.pages:
    text = page.extract_text()
    for line in text.split("\n"):
        line = line.strip()
        m = re.search(r"^(.*?)\s+(\d{4})$", line)
        if m:
            bureau = m.group(1).strip()
            cp = m.group(2)
            rows.append({"Ville": bureau, "Code Postal": cp})

pdf_tunisie.close()

df_tunisie = pd.DataFrame(rows)
df_tunisie["Pays"] = "Tunisie"                       # <-- AJOUT INDISPENSABLE
df_tunisie["Code Postal"] = df_tunisie["Code Postal"].astype(str)

df_tunisie = df_tunisie[["Ville", "Pays", "Code Postal"]]

print(df_tunisie.head())

                Ville     Pays Code Postal
0      Tunis Carthage  Tunisie        2035
1  Cité Ennasr Ariana  Tunisie        2001
2       Borj Baccouch  Tunisie        2027
3              Soukra  Tunisie        2036
4              Ariana  Tunisie        2080


**on a remarqué que dans le fichier que j ai importé il manque la ville de tunis**

In [84]:
df_tunisie = pd.concat([
    df_tunisie,
    pd.DataFrame([{"Ville": "Tunis", "Pays": "Tunisie", "Code Postal": "1000"}])
], ignore_index=True)

In [85]:
df_tunisie.to_excel("tunisie.xlsx", index=False)

# **Base de données extraite du Web contenant l'ensemble des villes de France et leurs codes postaux**

In [86]:
import pandas as pd

df_france = pd.read_excel("/content/Feuille de calcul sans titre.xlsx")

df_france = df_france[["ville", "Code_postal"]].rename(
    columns={"ville": "Ville", "Code_postal": "Code Postal"}
)

df_france["Pays"] = "France"                          # <-- AJOUT INDISPENSABLE
df_france["Code Postal"] = df_france["Code Postal"].astype(str).str.replace(r"\.0$", "", regex=True)

df_france = df_france.drop_duplicates()
df_france.to_excel("France_Villes_Codes.xlsx", index=False)
print(df_france.head())

                     Ville Code Postal    Pays
0  L ABERGEMENT CLEMENCIAT        1400  France
1    L ABERGEMENT DE VAREY        1640  France
2        AMBERIEU EN BUGEY        1500  France
3      AMBERIEUX EN DOMBES        1330  France
4                  AMBLEON        1300  France


# **Échantillon des données des villes de différents pay**

Plusieurs référentiels de villes ont été constitués (France, Tunisie, Espagne, Italie, Maroc, Belgique, Suisse, Algérie, Allemagne, Autriche, Pologne et Portugal). Ces référentiels ont ensuite été fusionnés dans une seule base de référence.

In [89]:


espagne = pd.DataFrame({
    "Ville": [
        "Madrid",
        "Barcelone",
        "Valence",
        "Séville",
        "Saragosse",
        "Malaga",
        "Murcie",
        "Palma",
        "Bilbao",
        "Alicante"
    ],
    "Pays": ["Espagne"] * 10,
    "Code Postal": [
        "28001",
        "08001",
        "46001",
        "41001",
        "50001",
        "29001",
        "30001",
        "07001",
        "48001",
        "03001"
    ]
})

espagne.to_excel("espagne_reference.xlsx", index=False)

# ---------------- ITALIE ---------------- #

italie = pd.DataFrame({
    "Ville": [
        "Rome",
        "Milan",
        "Naples",
        "Turin",
        "Florence",
        "Bologne",
        "Venise",
        "Palerme",
        "Gênes",
        "Bari"
    ],
    "Pays": ["Italie"] * 10,
    "Code Postal": [
        "00118",
        "20121",
        "80121",
        "10121",
        "50121",
        "40121",
        "30121",
        "90133",
        "16121",
        "70121"
    ]
})

italie.to_excel("italie_reference.xlsx", index=False)

# ---------------- MAROC ---------------- #

maroc = pd.DataFrame({
    "Ville": [
        "Casablanca",
        "Rabat",
        "Marrakech",
        "Fès",
        "Tanger",
        "Agadir",
        "Meknès",
        "Oujda",
        "Kénitra",
        "Tétouan"
    ],
    "Pays": ["Maroc"] * 10,
    "Code Postal": [
        "20000",
        "10000",
        "40000",
        "30000",
        "90000",
        "80000",
        "50000",
        "60000",
        "14000",
        "93000"
    ]
})

maroc.to_excel("maroc_reference.xlsx", index=False)

# ---------------- Belgique ---------------- #

import pandas as pd

belgique = pd.DataFrame({
    "Ville": ["Bruxelles", "Anvers", "Liège", "Gand", "Charleroi", "Bruges", "Namur", "Louvain", "Mons", "Ostende"],
    "Pays": ["Belgique"] * 10,
    "Code Postal": ["1000", "2000", "4000", "9000", "6000", "8000", "5000", "3000", "7000", "8400"],
})
belgique.to_excel("belgique_reference.xlsx", index=False)

# ---------------- Suisse ---------------- #

suisse = pd.DataFrame({
    "Ville": ["Genève", "Zurich", "Bâle", "Lausanne", "Berne", "Lucerne", "Lugano", "Fribourg", "Neuchâtel", "Sion"],
    "Pays": ["Suisse"] * 10,
    "Code Postal": ["1200", "8000", "4000", "1000", "3000", "6000", "6900", "1700", "2000", "1950"],
})
suisse.to_excel("suisse_reference.xlsx", index=False)

# ---------------- Algérie ---------------- #

algerie = pd.DataFrame({
    "Ville": ["Alger", "Oran", "Constantine", "Annaba", "Blida", "Sétif", "Tlemcen", "Béjaïa", "Sidi Bel Abbès", "Biskra"],
    "Pays": ["Algérie"] * 10,
    "Code Postal": ["16000", "31000", "25000", "23000", "9000", "19000", "13000", "6000", "22000", "7000"],
})
algerie.to_excel("algerie_reference.xlsx", index=False)

# ---------------- Allemagne ---------------- #

allemagne = pd.DataFrame({
    "Ville": ["Berlin", "Francfort", "Munich", "Hambourg", "Cologne", "Stuttgart", "Düsseldorf", "Dresde", "Leipzig", "Mayence"],
    "Pays": ["Allemagne"] * 10,
    "Code Postal": ["10115", "60311", "80331", "20095", "50667", "70173", "40213", "01067", "04109", "55116"],
})
allemagne.to_excel("allemagne_reference.xlsx", index=False)

# ---------------- Autriche ---------------- #

autriche = pd.DataFrame({
    "Ville": ["Vienne", "Graz", "Linz", "Salzbourg", "Innsbruck", "Klagenfurt", "Villach", "Wels", "Sankt Pölten", "Dornbirn"],
    "Pays": ["Autriche"] * 10,
    "Code Postal": ["1010", "8010", "4020", "5020", "6020", "9020", "9500", "4600", "3100", "6850"],
})
autriche.to_excel("autriche_reference.xlsx", index=False)

# ---------------- Pologne ---------------- #

pologne = pd.DataFrame({
    "Ville": ["Varsovie", "Cracovie", "Lodz", "Wroclaw", "Poznan", "Gdansk", "Szczecin", "Bydgoszcz", "Lublin", "Katowice"],
    "Pays": ["Pologne"] * 10,
    "Code Postal": ["00-001", "30-001", "90-001", "50-001", "60-001", "80-001", "70-001", "85-001", "20-001", "40-001"],
})
pologne.to_excel("pologne_reference.xlsx", index=False)

# ---------------- Portugal ---------------- #

portugal = pd.DataFrame({
    "Ville": ["Lisbonne", "Porto", "Braga", "Coimbra", "Faro", "Setubal", "Aveiro", "Evora", "Viseu", "Funchal"],
    "Pays": ["Portugal"] * 10,
    "Code Postal": ["1000", "4000", "4700", "3000", "8000", "2900", "3800", "7000", "3500", "9000"],
})
portugal.to_excel("portugal_reference.xlsx", index=False)

# Royaume-Uni
royaume_uni = pd.DataFrame({
    "Ville": ["Londres", "Manchester", "Birmingham", "Liverpool", "Leeds", "Bristol", "Glasgow", "Édimbourg", "Cardiff", "Belfast"],
    "Pays": ["Royaume-Uni"] * 10,
    "Code Postal": ["SW1A", "M1", "B1", "L1", "LS1", "BS1", "G1", "EH1", "CF10", "BT1"],
})
royaume_uni.to_excel("royaume_uni_reference.xlsx", index=False)

# Canada
canada = pd.DataFrame({
    "Ville": ["Montréal", "Toronto", "Ottawa", "Québec", "Vancouver", "Calgary", "Edmonton", "Winnipeg", "Halifax", "Hamilton"],
    "Pays": ["Canada"] * 10,
    "Code Postal": ["H1A", "M5H", "K1A", "G1A", "V5K", "T2P", "T5A", "R3C", "B3H", "L8P"],
})
canada.to_excel("canada_reference.xlsx", index=False)

# États-Unis
usa = pd.DataFrame({
    "Ville": ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphie", "San Antonio", "San Diego", "Dallas", "San José"],
    "Pays": ["États-Unis"] * 10,
    "Code Postal": ["10001", "90001", "60601", "77001", "85001", "19102", "78201", "92101", "75201", "95101"],
})
usa.to_excel("usa_reference.xlsx", index=False)

# Sénégal
senegal = pd.DataFrame({
    "Ville": ["Dakar", "Thiès", "Saint-Louis", "Kaolack", "Ziguinchor", "Diourbel", "Tambacounda", "Kolda", "Louga", "Fatick"],
    "Pays": ["Sénégal"] * 10,
    "Code Postal": ["10000", "21000", "32000", "40000", "27000", "25000", "46000", "37000", "29000", "23000"],
})
senegal.to_excel("senegal_reference.xlsx", index=False)

# Roumanie
roumanie = pd.DataFrame({
    "Ville": ["Bucarest", "Cluj-Napoca", "Timișoara", "Iași", "Constanța", "Brașov", "Craiova", "Oradea", "Sibiu", "Galați"],
    "Pays": ["Roumanie"] * 10,
    "Code Postal": ["010011", "400001", "300001", "700001", "900001", "500001", "200001", "410001", "550001", "800001"],
})
roumanie.to_excel("roumanie_reference.xlsx", index=False)

# Turquie
turquie = pd.DataFrame({
    "Ville": ["Istanbul", "Ankara", "Izmir", "Bursa", "Antalya", "Adana", "Konya", "Gaziantep", "Kayseri", "Mersin"],
    "Pays": ["Turquie"] * 10,
    "Code Postal": ["34000", "06000", "35000", "16000", "07000", "01000", "42000", "27000", "38000", "33000"],
})
turquie.to_excel("turquie_reference.xlsx", index=False)



In [90]:
!pip install rapidfuzz

## 5. Correction et enrichissement des lieux de naissance

L'objectif de cette étape est d'améliorer la qualité des informations contenues dans la colonne **Lieu de naissance**.

Les valeurs saisies peuvent présenter différents problèmes :

- fautes d'orthographe ;
- accents manquants ;
- espaces supplémentaires ;
- différentes façons d'écrire une même ville.

Par exemple :

| Valeur saisie | Ville attendue |
|---------------|----------------|
| Pariss | Paris |
| Marsseille | Marseille |
| Tuniss | Tunis |
| Casablanka | Casablanca |

### Construction d'un référentiel

Afin de corriger automatiquement ces erreurs, plusieurs référentiels de villes ont été construits.

Chaque référentiel contient :

- le nom de la ville ;
- le pays ;
- le ou les codes postaux associés.

Des référentiels ont été créés pour plusieurs pays (France, Tunisie, Espagne, Italie, Maroc, Belgique, Suisse, Algérie, Allemagne, Autriche, Pologne et Portugal), puis fusionnés dans une seule base de données.

Cette base sert de référence pendant toute la phase de correction.

---

### Normalisation des villes

Avant toute comparaison, les noms des villes sont normalisés.

Cette étape consiste à :

- convertir les caractères en minuscules ;
- supprimer les accents ;
- supprimer les espaces inutiles.

Ainsi, les écritures suivantes deviennent identiques :

```text
Túnis
TUNIS
 tunis
```

↓

```text
tunis
```

Cette normalisation évite que des différences d'écriture empêchent de retrouver une ville.

---

### Recherche de la meilleure correspondance

Une fois la ville normalisée, le script recherche la meilleure correspondance dans le référentiel.

Si la ville est correctement écrite, elle est retrouvée directement.

Si une faute d'orthographe est présente, le script utilise **RapidFuzz** afin de rechercher la ville la plus proche.

RapidFuzz calcule un **score de similarité** entre la valeur saisie et les villes présentes dans le référentiel.

Par exemple :

| Ville saisie | Ville trouvée | Score |
|--------------|---------------|------:|
| Pariss | Paris | 91 % |
| Marsseille | Marseille | 95 % |
| Tuniss | Tunis | 91 % |

Le script conserve uniquement les correspondances dont le score est suffisamment élevé afin d'éviter les corrections erronées.

---

### Gestion des ambiguïtés

Certaines villes peuvent être très proches les unes des autres.

Pour éviter des corrections incorrectes, le script calcule également la **distance de Levenshtein**, qui mesure le nombre minimal de modifications nécessaires pour passer d'un mot à un autre.

Si plusieurs villes obtiennent exactement la même distance, le script considère le résultat comme ambigu et ne réalise aucune correction automatique.

La ligne est alors signalée pour une vérification manuelle.

---

### Enrichissement des données

Une fois la ville identifiée, le script récupère automatiquement les informations associées dans le référentiel :

- le nom corrigé de la ville ;
- le pays ;
- le ou les codes postaux.

Lorsque plusieurs codes postaux existent pour une même ville (comme Paris), le script conserve l'ensemble des codes afin de ne perdre aucune information.

---

### Traçabilité

Toutes les corrections effectuées sont enregistrées dans une colonne **Commentaire**.

Cette colonne indique notamment :

- la ville d'origine ;
- la ville corrigée ;
- le score de similarité obtenu ;
- le pays détecté ;
- le ou les codes postaux récupérés.

Ainsi, chaque correction peut être vérifiée facilement et reste entièrement traçable.

In [91]:
import pandas as pd
import unicodedata
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein
import math

files = [
    "tunisie.xlsx",
    "France_Villes_Codes.xlsx",
    "espagne_reference.xlsx",
    "italie_reference.xlsx",
    "maroc_reference.xlsx",
    "belgique_reference.xlsx",
    "suisse_reference.xlsx",
    "algerie_reference.xlsx",
    "allemagne_reference.xlsx",
    "autriche_reference.xlsx",
    "pologne_reference.xlsx",
    "portugal_reference.xlsx",
]

references = []
for f in files:
    data = pd.read_excel(f, dtype={"Code Postal": str})  # éviter la conversion en float
    data.columns = [c.strip() for c in data.columns]
    assert {"Ville", "Pays", "Code Postal"}.issubset(data.columns), f"Colonnes manquantes dans {f}"
    references.append(data)

reference = pd.concat(references, ignore_index=True)
reference["Code Postal"] = (
    reference["Code Postal"].astype(str).str.replace(r"\.0$", "", regex=True)
)


def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return " ".join(text.split())


reference["Ville_norm"] = reference["Ville"].apply(normalize)
reference = reference[reference["Ville_norm"] != ""]   # on enlève les lignes vides

city_dict = {}
for _, row in reference.iterrows():
    ville = row["Ville_norm"]
    if ville not in city_dict:
        city_dict[ville] = {"Ville": row["Ville"], "Pays": row["Pays"], "Codes": set()}
    city_dict[ville]["Codes"].add(str(row["Code Postal"]))

city_names = list(city_dict.keys())

#####################################################
# Fonction principale
#####################################################


def corriger_ville(ville, score_threshold=80):
    ville_norm = normalize(ville)
    # Recherche exacte
    if ville_norm in city_dict:
      info = city_dict[ville_norm]
      return {
        "Ville corrigée": info["Ville"],
        "Pays": info["Pays"],
        "Codes Postaux": ", ".join(sorted(info["Codes"])),
        "Score": 100,
      }

# Sinon seulement, utiliser RapidFuzz
    vide = {"Ville corrigée": None, "Pays": None, "Codes Postaux": None, "Score": 0}
    if not ville_norm:
        return vide

    candidats = process.extract(ville_norm, city_names, scorer=fuzz.ratio, limit=5)
    if not candidats:
        return vide

    # distance max tolérée = proportionnelle à la longueur du mot
    # (évite "Alger" -> "GER" : 5 lettres, on ne tolère qu'1 lettre d'écart)
    max_dist = max(1, round(len(ville_norm) / 5))

    # on ne garde que les candidats de longueur proche (évite les comparaisons absurdes)
    candidats = [c for c in candidats if abs(len(c[0]) - len(ville_norm)) <= max_dist + 1]
    if not candidats:
        return vide

    meilleur = min(candidats, key=lambda c: Levenshtein.distance(ville_norm, c[0]))
    match, score, _ = meilleur
    distance = Levenshtein.distance(ville_norm, match)

    # égalité entre plusieurs candidats à distance minimale -> ambigu, on n'invente rien
    ex_aequo = [c for c in candidats if Levenshtein.distance(ville_norm, c[0]) == distance]
    if len(ex_aequo) > 1 and distance > 0:
        return {**vide, "Score": score}  # ambigu : à vérifier manuellement

    if score < score_threshold and distance > max_dist:
        return {**vide, "Score": score}

    info = city_dict[match]
    return {
        "Ville corrigée": info["Ville"],
        "Pays": info["Pays"],
        "Codes Postaux": ", ".join(sorted(info["Codes"])),
        "Score": score,
    }

In [92]:
for v in ["Pariss", "Marsseille", "Tuniss", "Casablanka", "Lyone", "Bordeau"]:
    print(v, "->", corriger_ville(v))

Pariss -> {'Ville corrigée': 'PARIS', 'Pays': 'France', 'Codes Postaux': '75001, 75002, 75003, 75004, 75005, 75006, 75007, 75008, 75009, 75010, 75011, 75012, 75013, 75014, 75015, 75016, 75017, 75018, 75019, 75020, 75116', 'Score': 90.9090909090909}
Marsseille -> {'Ville corrigée': 'MARSEILLE', 'Pays': 'France', 'Codes Postaux': '13001, 13002, 13003, 13004, 13005, 13006, 13007, 13008, 13009, 13010, 13011, 13012, 13013, 13014, 13015, 13016', 'Score': 94.73684210526316}
Tuniss -> {'Ville corrigée': 'Tunis', 'Pays': 'Tunisie', 'Codes Postaux': '1000', 'Score': 90.9090909090909}
Casablanka -> {'Ville corrigée': 'Casablanca', 'Pays': 'Maroc', 'Codes Postaux': '20000', 'Score': 90.0}
Lyone -> {'Ville corrigée': 'LYON', 'Pays': 'France', 'Codes Postaux': '69001, 69002, 69003, 69004, 69005, 69006, 69007, 69008, 69009', 'Score': 88.88888888888889}
Bordeau -> {'Ville corrigée': None, 'Pays': None, 'Codes Postaux': None, 'Score': 93.33333333333333}


In [93]:
print(corriger_ville("Bordeau"))
print(corriger_ville("Pariss"))
print(corriger_ville("Marsseille"))

{'Ville corrigée': None, 'Pays': None, 'Codes Postaux': None, 'Score': 93.33333333333333}
{'Ville corrigée': 'PARIS', 'Pays': 'France', 'Codes Postaux': '75001, 75002, 75003, 75004, 75005, 75006, 75007, 75008, 75009, 75010, 75011, 75012, 75013, 75014, 75015, 75016, 75017, 75018, 75019, 75020, 75116', 'Score': 90.9090909090909}
{'Ville corrigée': 'MARSEILLE', 'Pays': 'France', 'Codes Postaux': '13001, 13002, 13003, 13004, 13005, 13006, 13007, 13008, 13009, 13010, 13011, 13012, 13013, 13014, 13015, 13016', 'Score': 94.73684210526316}


In [94]:
print(city_names[:20])

['tunis carthage', 'cite ennasr ariana', 'borj baccouch', 'soukra', 'ariana', 'ariana geant', 'menzah 6', 'cite la gazelle', 'mjaz elbab', 'teboursouk', 'beja', 'dougga', 'rades medina', 'mhamdia', 'hammam lif', 'rades', 'ezzahra', 'ben arous', 'errisala', 'ezzahra el habib']


In [95]:
print("paris" in city_names)

True


In [96]:
def traiter_ligne(row):
    commentaires = []

    nom, prenom, c = split_nom_prenom(row["NOM"], row["PRÉNOM"])
    if c != "NOM/PRÉNOM déjà séparés.":
        commentaires.append(c)
    prenom = prenom.strip().title() if prenom else prenom

    date_corrigee, c = parse_date(row["DATE DE NAISSANCE"])
    if "déjà au bon format" not in c:
        commentaires.append(c)

    lieu_brut = row["LIEU DE NAISSANCE"]
    if pd.isna(lieu_brut) or str(lieu_brut).strip() == "":
        info = {"Ville corrigée": None, "Pays": None, "Codes Postaux": None, "Score": 0}
    else:
        info = corriger_ville(lieu_brut)
        if info["Ville corrigée"] is not None:
            if normalize(lieu_brut) != normalize(info["Ville corrigée"]):
                commentaires.append(
                    f'Lieu corrigé : "{lieu_brut}" → "{info["Ville corrigée"]}" '
                    f'(score {info["Score"]:.1f})'
                )
            commentaires.append(f'Pays détecté : {info["Pays"]}')
            commentaires.append(f'Code postal récupéré : {info["Codes Postaux"]}')
        else:
            commentaires.append(f'Lieu de naissance "{lieu_brut}" non reconnu dans le référentiel.')

    return pd.Series({
        "NOM_corrigé": nom,
        "PRÉNOM_corrigé": prenom,
        "Date de naissance corrigé": date_corrigee,
        "LIEU CORRIGÉ": info["Ville corrigée"],
        "PAYS": info["Pays"],
        "CODE POSTAL": info["Codes Postaux"],
        "Commentaire": " | ".join(commentaires),
    })

In [97]:
print(df.columns.tolist())

['ID', 'NOM', 'PRÉNOM', 'DATE DE NAISSANCE', 'LIEU DE NAISSANCE', 'NOM_corrigé', 'PRÉNOM_corrigé', 'Date de naissance corrigé', 'LIEU CORRIGÉ', 'PAYS', 'CODE POSTAL', 'Commentaire', 'Unnamed: 12']


In [98]:
corrections = df.apply(traiter_ligne, axis=1)

for col in corrections.columns:
    df[col] = corrections[col]

df.to_excel("Donnees_corrigees.xlsx", index=False)

In [99]:
df.head()

,ID,NOM,PRÉNOM,DATE DE NAISSANCE,LIEU DE NAISSANCE,NOM_corrigé,PRÉNOM_corrigé,Date de naissance corrigé,LIEU CORRIGÉ,PAYS,CODE POSTAL,Commentaire,Unnamed: 12
0,1,MARTIN,Jean,12/03/1842,Marseile,MARTIN,Jean,12/03/1842,MARSEILLE,France,"13001, 13002, 13003, 13004, 13005, 13006, 1300...","Lieu corrigé : ""Marseile"" → ""MARSEILLE"" (score...",Marseille
1,2,DUPONT Marie,NaN,05-07-1835,paris,DUPONT,Marie,05/07/1835,PARIS,France,"75001, 75002, 75003, 75004, 75005, 75006, 7500...",PRÉNOM manquant dans la colonne C : déduit du ...,NaN
2,3,BERNARD Pierre,NaN,1848-11-20,BORDEAU,BERNARD,Pierre,20/11/1848,None,None,None,PRÉNOM manquant dans la colonne C : déduit du ...,NaN
3,4,MOREAU Jeanne,NaN,09/09/1860,Naples,MOREAU,Jeanne,09/09/1860,Naples,Italie,80121,PRÉNOM manquant dans la colonne C : déduit du ...,NaN
4,5,SIMON Louis,NaN,22/04/1851,Lyone,SIMON,Louis,22/04/1851,LYON,France,"69001, 69002, 69003, 69004, 69005, 69006, 6900...",PRÉNOM manquant dans la colonne C : déduit du ...,NaN


## Limites de l'approche basée sur RapidFuzz

RapidFuzz permet de corriger efficacement la majorité des fautes d'orthographe simples en recherchant la ville la plus proche dans le référentiel.

Par exemple :

| Saisie | Correction |
|--------|------------|
| Pariss | Paris |
| Marsseille | Marseille |
| Tuniss | Tunis |
| Casablanka | Casablanca |

Cependant, cette méthode présente certaines limites.

### 1. Ambiguïtés

Lorsque plusieurs villes sont très proches, il peut être difficile de déterminer automatiquement laquelle est la bonne.

Par exemple, plusieurs villes peuvent avoir un nom très similaire ou différer uniquement par quelques caractères.

Dans ce cas, le script préfère ne pas effectuer de correction plutôt que de proposer une ville incorrecte.

---

### 2. Fautes trop importantes

Si le nom saisi est très éloigné de la ville réelle, le score de similarité devient insuffisant.

Exemple :

```text
Bdx
```

↓

Impossible de déterminer automatiquement qu'il s'agit de **Bordeaux**.

Le script considère alors que la correction n'est pas suffisamment fiable.

---

### 3. Référentiel incomplet

La qualité des résultats dépend directement du référentiel utilisé.

Si une ville n'est pas présente dans la base de référence, aucune correction automatique ne peut être proposée, même si le nom est correctement écrit.

---

### 4. Cas particuliers

Certaines écritures restent difficiles à corriger automatiquement, notamment :

- les abréviations (`St Denis` au lieu de `Saint-Denis`) ;
- les noms historiques ou anciens ;
- les erreurs comportant plusieurs fautes importantes ;
- les villes composées dont une partie est absente.

Ces situations nécessitent généralement une vérification manuelle.

---

### Perspectives d'amélioration

Une amélioration possible serait d'ajouter une seconde étape basée sur un modèle d'intelligence artificielle ou un modèle de langage (LLM).

L'idée serait de conserver RapidFuzz comme première méthode de correction, puis de faire appel au modèle uniquement lorsque celui-ci ne trouve aucune correspondance suffisamment fiable.

Cette approche permettrait d'améliorer la correction de certains cas complexes tout en conservant la rapidité de la recherche dans le référentiel.

In [100]:
!pip install openai

In [101]:
import os

os.environ["GROQ_API_KEY"] = "mettre une clé API"

In [102]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [103]:
import pandas as pd
import unicodedata
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein
import math

files = [
    "tunisie.xlsx",
    "France_Villes_Codes.xlsx",
    "espagne_reference.xlsx",
    "italie_reference.xlsx",
    "maroc_reference.xlsx",
    "belgique_reference.xlsx",
    "suisse_reference.xlsx",
    "algerie_reference.xlsx",
    "allemagne_reference.xlsx",
    "autriche_reference.xlsx",
    "pologne_reference.xlsx",
    "portugal_reference.xlsx",
]

references = []
for f in files:
    data = pd.read_excel(f, dtype={"Code Postal": str})  # éviter la conversion en float
    data.columns = [c.strip() for c in data.columns]
    assert {"Ville", "Pays", "Code Postal"}.issubset(data.columns), f"Colonnes manquantes dans {f}"
    references.append(data)

reference = pd.concat(references, ignore_index=True)
reference["Code Postal"] = (
    reference["Code Postal"].astype(str).str.replace(r"\.0$", "", regex=True)
)


def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return " ".join(text.split())


reference["Ville_norm"] = reference["Ville"].apply(normalize)
reference = reference[reference["Ville_norm"] != ""]   # on enlève les lignes vides

city_dict = {}
for _, row in reference.iterrows():
    ville = row["Ville_norm"]
    if ville not in city_dict:
        city_dict[ville] = {"Ville": row["Ville"], "Pays": row["Pays"], "Codes": set()}
    city_dict[ville]["Codes"].add(str(row["Code Postal"]))

city_names = list(city_dict.keys())

#####################################################
# Fonction principale
#####################################################
def corriger_ville_llm(ville):

    prompt = f"""
Tu es un expert en géographie.

Corrige uniquement le nom de la ville.

Retourne uniquement le nom corrigé.

Ville :
{ville}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    ville_corrigee = response.choices[0].message.content.strip()

    ville_norm = normalize(ville_corrigee)

    if ville_norm in city_dict:

        info = city_dict[ville_norm]

        return {
            "Ville corrigée": info["Ville"],
            "Pays": info["Pays"],
            "Codes Postaux": ", ".join(sorted(info["Codes"])),
            "Score": 100
        }

    return {
        "Ville corrigée": None,
        "Pays": None,
        "Codes Postaux": None,
        "Score": 0
    }

In [104]:
def traiter_ligne(row):
    commentaires = []

    nom, prenom, c = split_nom_prenom(row["NOM"], row["PRÉNOM"])
    if c != "NOM/PRÉNOM déjà séparés.":
        commentaires.append(c)
    prenom = prenom.strip().title() if prenom else prenom

    date_corrigee, c = parse_date(row["DATE DE NAISSANCE"])
    if "déjà au bon format" not in c:
        commentaires.append(c)

    lieu_brut = row["LIEU DE NAISSANCE"]
    if pd.isna(lieu_brut) or str(lieu_brut).strip() == "":
        info = {"Ville corrigée": None, "Pays": None, "Codes Postaux": None, "Score": 0}
    else:
        info = corriger_ville_llm(lieu_brut)
        if info["Ville corrigée"] is not None:
            if normalize(lieu_brut) != normalize(info["Ville corrigée"]):
                commentaires.append(
                    f'Lieu corrigé : "{lieu_brut}" → "{info["Ville corrigée"]}" '
                    f'(score {info["Score"]:.1f})'
                )
            commentaires.append(f'Pays détecté : {info["Pays"]}')
            commentaires.append(f'Code postal récupéré : {info["Codes Postaux"]}')
        else:
            commentaires.append(f'Lieu de naissance "{lieu_brut}" non reconnu dans le référentiel.')

    return pd.Series({
        "NOM_corrigé": nom,
        "PRÉNOM_corrigé": prenom,
        "Date de naissance corrigé": date_corrigee,
        "LIEU CORRIGÉ": info["Ville corrigée"],
        "PAYS": info["Pays"],
        "CODE POSTAL": info["Codes Postaux"],
        "Commentaire": " | ".join(commentaires),
    })

In [105]:
corrections = df.apply(traiter_ligne, axis=1)

for col in corrections.columns:
    df[col] = corrections[col]

df.to_excel("Donnees_corrigees1.xlsx", index=False)

In [106]:
df.head()

,ID,NOM,PRÉNOM,DATE DE NAISSANCE,LIEU DE NAISSANCE,NOM_corrigé,PRÉNOM_corrigé,Date de naissance corrigé,LIEU CORRIGÉ,PAYS,CODE POSTAL,Commentaire,Unnamed: 12
0,1,MARTIN,Jean,12/03/1842,Marseile,MARTIN,Jean,12/03/1842,MARSEILLE,France,"13001, 13002, 13003, 13004, 13005, 13006, 1300...","Lieu corrigé : ""Marseile"" → ""MARSEILLE"" (score...",Marseille
1,2,DUPONT Marie,NaN,05-07-1835,paris,DUPONT,Marie,05/07/1835,PARIS,France,"75001, 75002, 75003, 75004, 75005, 75006, 7500...",PRÉNOM manquant dans la colonne C : déduit du ...,NaN
2,3,BERNARD Pierre,NaN,1848-11-20,BORDEAU,BERNARD,Pierre,20/11/1848,BORDEAUX,France,"33000, 33100, 33200, 33300, 33800",PRÉNOM manquant dans la colonne C : déduit du ...,NaN
3,4,MOREAU Jeanne,NaN,09/09/1860,Naples,MOREAU,Jeanne,09/09/1860,Naples,Italie,80121,PRÉNOM manquant dans la colonne C : déduit du ...,NaN
4,5,SIMON Louis,NaN,22/04/1851,Lyone,SIMON,Louis,22/04/1851,LYON,France,"69001, 69002, 69003, 69004, 69005, 69006, 6900...",PRÉNOM manquant dans la colonne C : déduit du ...,NaN


### Conclusion

### Limites de l'approche

Malgré l'utilisation d'un référentiel de villes, de RapidFuzz et d'un LLM, certaines valeurs n'ont pas pu être corrigées automatiquement.

En effet, plusieurs entrées du jeu de données ne correspondent pas à des **villes**, mais à d'autres types de lieux géographiques ou administratifs. Par exemple :

- **France** : il s'agit d'un pays et non d'une ville.
- **Côte-d'Or** : il s'agit d'un département français et non d'une ville.
- **Royaume de Prusse** : il s'agit d'un ancien État européen historique et non d'une ville.

L'objectif du pipeline étant de reconnaître uniquement des villes afin d'associer un pays et un code postal, ces valeurs ne peuvent pas être enrichies automatiquement.

Par ailleurs, certaines villes possèdent plusieurs codes postaux. C'est notamment le cas de **Paris**, dont les différents arrondissements sont associés à des codes postaux distincts (75001 à 75020). Afin de ne pas perdre d'information, le référentiel conserve l'ensemble des codes postaux associés à une même ville et les retourne sous forme de liste lorsqu'une correspondance est trouvée.

Dans ces cas particuliers, une intervention manuelle ou l'utilisation d'un référentiel géographique plus complet (incluant les pays, les régions, les départements et les anciennes entités administratives) permettrait d'améliorer davantage la qualité de l'enrichissement des données.